# **News classification advanced ML final project**
Text Classification | Tengri News (Kazakh)
### **Team TriA: Aknur Mansurkhan, Akbike Zhurymbay, Aruzhan Iliyas**

### **IDs: 230107148, 230107053, 230107049**

________________________

**DATASETS**
--------
1. **tengri_news.csv**         – Kaggle, ~19 k articles, Kazakh language
   Source: https://www.kaggle.com/datasets/kuantaiulysalamat/kazakh-news-articles-dataset
2. **tengri-kaz-parsed.csv**   – Manually parsed from tengri.kz (2024-2026),
   Kazakh language, includes publication date


### **1-Data loading ,  Aknur**

In [1]:
import os
import ast
import pandas as pd

**# load Kaggle dataset**

In [2]:
df_kaggle = pd.read_csv("tengri_news.csv", engine='python', on_bad_lines='warn', encoding='utf-8', encoding_errors='replace')
df_kaggle["source"] = "kaggle"
df_kaggle["date"]   = pd.NA

/tmp/ipykernel_516/3163429993.py:1: ParserWarning: Skipping line 16232: unexpected end of data

  df_kaggle = pd.read_csv("tengri_news.csv", engine='python', on_bad_lines='warn', encoding='utf-8', encoding_errors='replace')


In [3]:
print(f"shape       : {df_kaggle.shape}")
print(f"columns     : {df_kaggle.columns.tolist()}")
print(f"null counts :{df_kaggle.isnull().sum()}")

shape       : (16230, 6)
columns     : ['title', 'url', 'tags', 'text', 'source', 'date']
null counts :title         0
url           0
tags          0
text          0
source        0
date      16230
dtype: int64


In [4]:
df_kaggle.head()

,title,url,tags,text,source,date
0,Әлемдегі ең танымал туристік бағыттар аталды,https://kaz.tengritravel.kz/around-the-world/a...,"['туризм', 'рейтинг', 'саяхат']",Tripadvisor нұсқасы бойынша Индонезияның Бали ...,kaggle,<NA>
1,2020 жылдың қорытындысы бойынша Қазақстан халқ...,https://kaz.tengrinews.kz/kazakhstan_news/2020...,"['қазақстан', 'халық саны']",2021 жылы елдегі халық саны 18 миллион 877 мың...,kaggle,<NA>
2,Қазақстандықтар 2021 жылды қарсы алып жатыр,https://kaz.tengrinews.kz/kazakhstan_news/kaza...,"['мереке', 'жаңа жыл']",Қазақстандықтар 2021 жылды қарсы алып жатыр. 2...,kaggle,<NA>
3,2022 жылы Балқаш көлінде демалу қанша тұрады,https://kaz.tengritravel.kz/my-country/2022-jy...,"['туризм', 'балқаш', 'турист', 'саяхат']",2022 жылдың жазғы маусымында Балқаш көлінде де...,kaggle,<NA>
4,Тоқаев қазақстандықтарды 2022 жылы не күтіп тұ...,https://kaz.tengrinews.kz/kazakhstan_news/toka...,"['тоқаев қасым-жомарт', 'жаңа жыл']",Президент Қасым-Жомарт Тоқаев қазақстандықтард...,kaggle,<NA>


**# load manually parsed dataset**

In [5]:
df_parsed = pd.read_csv("tengri-kaz-parsed.csv", engine='python', on_bad_lines='warn', encoding='utf-8', encoding_errors='replace')
df_parsed["source"] = "parsed"


In [6]:
print(f"shape       : {df_parsed.shape}")
print(f"columns     : {df_parsed.columns.tolist()}")
print(f"null counts :\n{df_parsed.isnull().sum()}")

shape       : (12619, 6)
columns     : ['title', 'url', 'tags', 'text', 'date', 'source']
null counts :
title         0
url           0
tags      12619
text         22
date          0
source        0
dtype: int64


In [7]:
df_parsed.head()

,title,url,tags,text,date,source
0,"Астана, Алматы және Шымкентте есірткі тұтынушы...",https://kaz.tengrinews.kz/kazakhstan_news/asta...,NaN,Қазақстанда ағын суларда есірткінің бар-жоғы з...,Бүгін 17:25,parsed
1,“Ешқандай таныс ықпал ете алмайды“ - вице-мини...,https://kaz.tengrinews.kz/kazakhstan_news/eshk...,NaN,Ішкі істер министрінің орынбасары Санжар Әділо...,Бүгін 16:52,parsed
2,“Сіз iPhone сатып алдыңыз“: қазақстандықтарға ...,https://kaz.tengrinews.kz/kazakhstan_news/sz-i...,NaN,Бас прокуратура қазақстандықтарға PayPal төлем...,Бүгін 16:28,parsed
3,Астанада Парламент палаталарының бірлескен оты...,https://kaz.tengrinews.kz/kazakhstan_news/asta...,NaN,Мәжіліс төрағасы Ерлан Қошанов Парламент палат...,Бүгін 16:17,parsed
4,Астана мен Алматыда 14-16 мамырда ауа райы қан...,https://kaz.tengrinews.kz/kazakhstan_news/asta...,NaN,"""Қазгидромет"" синоптиктері Астана мен Алматыда...",Бүгін 16:16,parsed


## **# combine datasets for preprocess (at the end we will separate them again)**

In [8]:
COMMON_COLS = ["title", "url", "tags", "text", "date", "source"]
for col in COMMON_COLS:
    if col not in df_kaggle.columns:
        df_kaggle[col] = pd.NA
    if col not in df_parsed.columns:
        df_parsed[col] = pd.NA

df_kaggle = df_kaggle[COMMON_COLS]
df_parsed  = df_parsed[COMMON_COLS]
df_combined = pd.concat([df_kaggle, df_parsed], ignore_index=True)


In [9]:
print(f"combined shape  : {df_combined.shape}")
print(f"source breakdown:\n{df_combined['source'].value_counts()}")
print(f"\noverall null counts:\n{df_combined.isnull().sum()}")

combined shape  : (28849, 6)
source breakdown:
source
kaggle    16230
parsed    12619
Name: count, dtype: int64

overall null counts:
title         0
url           0
tags      12619
text         22
date      16230
source        0
dtype: int64


## **# cleaning steps**

In [10]:
empty_rows = df_combined[df_combined["text"].isna() & df_combined["title"].isna()]
print(f"rows with both title AND text empty : {len(empty_rows)}")

rows with both title AND text empty : 0


In [11]:
# Text length distribution
df_combined["text_len"] = df_combined["text"].fillna("").apply(len)
print("\nText length statistics (characters):")
print(df_combined["text_len"].describe().round(1))


Text length statistics (characters):
count    28849.0
mean      1540.4
std       1497.7
min          0.0
25%        919.0
50%       1252.0
75%       1747.0
max      80872.0
Name: text_len, dtype: float64


In [12]:
# tag coverage in Kaggle set
kaggle_mask    = df_combined["source"] == "kaggle"
tags_available = df_combined.loc[kaggle_mask, "tags"].notna().sum()
print(f"\nKaggle rows with tags : {tags_available} / {kaggle_mask.sum()}")


Kaggle rows with tags : 16230 / 16230


In [13]:
# Duplicate URLs
dup_urls = df_combined.duplicated(subset=["url"]).sum()
print(f"Duplicate URLs        : {dup_urls}")

Duplicate URLs        : 291


In [14]:
df_combined = df_combined.drop_duplicates(subset=['url']).reset_index(drop=True)
print(f"Duplicate URLs after removal: {df_combined.duplicated(subset=['url']).sum()}")

Duplicate URLs after removal: 0


# **2. Preprocessing & feature engineering**

In [15]:
CATEGORY_MAP = {
    "politics": [                          # Politics
        "тоқаев", "ақорда", "парламент", "үкімет", "министр", "президент",
        "мажилис", "сенат", "сайлау", "тағайындау", "дипломатия", "министрлік",
        "әкім", "акорда", "саясат", "билік",
    ],
    "health": [                       # Health
        "коронавирус", "вакцина", "пандемия", "пневмония", "карантин",
        "дәрі", "медицина", "ауруxана", "дәрігер", "денсаулық", "ковид",
        "covid", "санитар",
    ],
    "crime": [                          # Crime / Law
        "қылмыс", "полиция", "ұрлық", "тұтқын", "сот", "прокурор",
        "абақты", "құқық қорғау", "кісі өлтіру", "зорлау", "жазалау",
        "тергеу", "алаяқтық", "бандит",
    ],
    "sports": [                           # Sports
        "спорт", "бокс", "жекпе-жек", "футбол", "олимпиада", "чемпион",
        "турнир", "жарыс", "алтын медаль", "күресші", "велоспорт",
        "хоккей", "теннис",
    ],
    "economy": [                       # Economy / Business
        "экономика", "банк", "теңге", "инфляция", "бюджет", "инвестиция",
        "компания", "бизнес", "нарық", "мұнай", "газ", "өнеркәсіп",
        "импорт", "экспорт", "жұмыссыздық", "жалақы",
    ],
    "education": [                           # Education / Science
        "мектеп", "университет", "студент", "оқушы", "білім", "ент",
        "ғылым", "зерттеу", "профессор", "колледж", "оқу",
    ],
    "international": [                     # International
        "ресей", "ақш", "украина", "қытай", "еуропа", "бм", "дүниежүзі",
        "шетел", "халықаралық", "нато", "одақ", "елші", "саммит",
    ],
    "social": [                      # Social
        "балалар", "отбасы", "зейнеткер", "тұрғын үй", "азық-түлік",
        "жәрдемақы", "зейнетақы", "жетім", "кедейлік", "қоғам",
        "мүгедек",
    ],
}


In [16]:
df = df_combined.copy()

In [17]:
before = len(df)
df = df.drop_duplicates(subset=["url"])
print(f"Removed by duplicate URL  : {before - len(df):,} rows")


Removed by duplicate URL  : 0 rows


In [18]:
before = len(df)
df = df.drop_duplicates(subset=["title"])
print(f"Removed by duplicate title: {before - len(df):,} rows")

Removed by duplicate title: 793 rows


In [19]:
print(f"After dedup               : {len(df):,} rows")

After dedup               : 27,765 rows


In [20]:
before = len(df)
df = df[df["text"].notna() & df["text"].str.strip().ne("")]
df = df[df["title"].notna() & df["title"].str.strip().ne("")]
print(f"Removed empty rows: {before - len(df):,}")
print(f"Remaining         : {len(df):,}")

Removed empty rows: 22
Remaining         : 27,743


# **3. Normalising Kazakh text**

In [21]:
def clean_kazakh(text: str) -> str:
    """Light normalisation that preserves Kazakh-specific characters
       Does NOT remove them – models need them"""
    if not isinstance(text, str):
        return ""
    import unicodedata
    import re
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"[\u00a0\u200b\u200c\u200d\ufeff]", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

In [22]:
df["title_clean"] = df["title"].apply(clean_kazakh)
df["text_clean"]  = df["text"].apply(clean_kazakh)

In [23]:
# Quick sanity check
sample_original = df["text"].iloc[0][:120]
sample_cleaned  = df["text_clean"].iloc[0][:120]
print(f"Original sample : {sample_original!r}")
print(f"Cleaned  sample : {sample_cleaned!r}")

Original sample : 'Tripadvisor нұсқасы бойынша Индонезияның Бали аралы ең танымал туристік бағыт болып танылды, - деп хабарлайды Kaz.tengri'
Cleaned  sample : 'Tripadvisor нұсқасы бойынша Индонезияның Бали аралы ең танымал туристік бағыт болып танылды, - деп хабарлайды Kaz.tengri'


# **4. Parsing tag strings**

In [24]:
def parse_tags(raw) -> list:
    if pd.isna(raw):
        return []
    if isinstance(raw, list):
        return raw
    try:
        parsed = ast.literal_eval(str(raw))
        if isinstance(parsed, list):
            return [t.strip().lower() for t in parsed]
    except Exception:
        pass
    return [t.strip().lower() for t in str(raw).split(",") if t.strip()]

df["tags_list"] = df["tags"].apply(parse_tags)
has_tags = df["tags_list"].apply(len) > 0
print(f"Rows with >=1 tag : {has_tags.sum():,} / {len(df):,}")

Rows with >=1 tag : 16,229 / 27,743


# **5. Assigning category labels**

In [25]:
def assign_category(tags_list: list) -> str:
    joined = " ".join(tags_list)  # single string to search
    for cat, keywords in CATEGORY_MAP.items():
        for kw in keywords:
            if kw in joined:
                return cat
    return "other"

df["category"] = df["tags_list"].apply(assign_category)
print("Category distribution:")
print(df["category"].value_counts().to_string())

Category distribution:
category
other            18046
health            2854
politics          1953
sports            1242
crime             1152
international      912
economy            642
education          528
social             414


In [26]:
# For training we keep only rows with a meaningful category
before = len(df)
df_labeled   = df[df["category"] != "other"].copy()
df_unlabeled = df[df["category"] == "other"].copy()
print(f"\nLabeled rows   : {len(df_labeled):,}")
print(f"Unlabeled rows : {len(df_unlabeled):,}")
print("(Unlabeled rows are kept in preprocessed.csv with category='other')")


Labeled rows   : 9,697
Unlabeled rows : 18,046
(Unlabeled rows are kept in preprocessed.csv with category='other')


# **6. Adding basic text-length features**

In [27]:
for d in [df_labeled, df_unlabeled]:
    d["char_count"]  = d["text_clean"].apply(len)
    d["word_count"]  = d["text_clean"].apply(lambda x: len(x.split()))
    d["title_len"]   = d["title_clean"].apply(len)

df_all = pd.concat([df_labeled, df_unlabeled], ignore_index=True)
print(df_all[["char_count", "word_count", "title_len"]].describe().round(1))

       char_count  word_count  title_len
count     27743.0     27743.0    27743.0
mean       1550.3       192.8       61.6
std        1510.0       188.9       15.2
min         186.0        24.0        4.0
25%         933.0       115.0       51.0
50%        1261.0       156.0       61.0
75%        1754.5       218.0       72.0
max       80872.0      8997.0      146.0


# **7. Normalising date column (parsed subset)**

In [28]:
KAZ_MONTHS = {
    "қаңтар": "01", "ақпан": "02", "наурыз": "03",
    "сәуір": "04",  "мамыр": "05", "маусым": "06",
    "шілде": "07",  "тамыз": "08", "қыркүйек": "09",
    "қазан": "10",  "қараша": "11", "желтоқсан": "12",
}

In [29]:
import re

In [30]:
def parse_date(raw) -> str:
    if pd.isna(raw):
        return pd.NA
    s = str(raw).strip().lower()
    if s.startswith("бүгін"):     # "Today HH:MM"
        return "2026-05-13"       # replace with current scrape date
    if s.startswith("кеше"):      # "Yesterday HH:MM"
        return "2026-05-12"
    # Try "DD Month" or "DD Month YYYY"
    m = re.search(r"(\d{1,2})\s+([а-яёa-zқғүіңөәһ]+)\s*(\d{4})?", s)
    if m:
        day  = m.group(1).zfill(2)
        mon  = KAZ_MONTHS.get(m.group(2), "00")
        year = m.group(3) or "2025"
        return f"{year}-{mon}-{day}"
    return pd.NA

In [31]:
df_all["date_parsed"] = df_all["date"].apply(parse_date)
parsed_dates = df_all["date_parsed"].notna().sum()
print(f"Rows with parsed date : {parsed_dates:,} / {len(df_all):,}")
print("Sample parsed dates:")
print(df_all[["date", "date_parsed"]].dropna(subset=["date"]).head(6).to_string(index=False))

Rows with parsed date : 11,513 / 27,743
Sample parsed dates:
       date date_parsed
Бүгін 17:25  2026-05-13
Бүгін 16:52  2026-05-13
Бүгін 16:28  2026-05-13
Бүгін 16:17  2026-05-13
Бүгін 16:16  2026-05-13
Бүгін 16:02  2026-05-13


## **keep cleaned features**

In [32]:
KEEP = [
    "title_clean",
    "text_clean",
    "url",
    "source",
    "tags_list",
    "category",
    "char_count",
    "word_count",
    "title_len",
    "date_parsed",
]

In [33]:
df = df_all[KEEP].copy()
df.columns = [
    "title", "text", "url", "source",
    "tags", "category",
    "char_count", "word_count", "title_len",
    "date",
]

In [34]:
print(f"Shape  : {df.shape}")
print(f"\nFinal category breakdown:")
print(df["category"].value_counts().to_string())

Shape  : (27743, 10)

Final category breakdown:
category
other            18046
health            2854
politics          1953
sports            1242
crime             1152
international      912
economy            642
education          528
social             414


In [35]:
df.to_csv('preprocessed.csv',index=False, encoding="utf-8")